# Aro · Procesador de rendimiento físico

Este cuaderno toma la **grabación panorámica** de un partido (la vista completa de Veo grabada con OBS) y el **respaldo del partido** exportado desde Aro, y genera `fisico.json` con la distancia, la velocidad media y máxima, la alta intensidad y los sprints de cada jugador, más las posiciones para el minimapa.

**Antes de empezar**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** y Guardar.
2. En tu Google Drive creá la carpeta **Aro** y subí:
   - la grabación panorámica (por ejemplo `panoramica_mitre.mp4`);
   - el respaldo del partido: en Aro, **Análisis → Exportar → Respaldo completo (.json)**.
3. Completá los nombres en el paso 2 y tocá **Entorno de ejecución → Ejecutar todas**.

Si Colab se desconecta (pasa en sesiones largas), volvé a **Ejecutar todas**: retoma donde quedó, porque el avance se guarda en tu Drive.

## 1. Preparar

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SIN GPU: cambiá el entorno de ejecución a GPU antes de seguir"
!pip -q install ultralytics
!rm -rf /content/aro && git clone -q --depth 1 https://github.com/misionessporttech-collab/aro /content/aro
from google.colab import drive
drive.mount('/content/drive')

## 2. Datos del partido

In [ ]:
#@title Completá y ejecutá
VIDEO_PANORAMICA = "/content/drive/MyDrive/Aro/panoramica_mitre.mp4" #@param {type:"string"}
RESPALDO_ARO = "/content/drive/MyDrive/Aro/respaldo_mitre.json" #@param {type:"string"}
CARPETA_SALIDA = "/content/drive/MyDrive/Aro/fisico_mitre" #@param {type:"string"}
#@markdown Opcional: segundo del video del partido (el de YouTube) en que arranca la grabación panorámica. Dejalo en -1 para que lo busque solo.
INICIO_APROX = -1 #@param {type:"number"}
import os
assert os.path.exists(VIDEO_PANORAMICA), "No encuentro el video: revisá el nombre y la carpeta"
assert os.path.exists(RESPALDO_ARO), "No encuentro el respaldo de Aro: revisá el nombre y la carpeta"
os.makedirs(CARPETA_SALIDA, exist_ok=True)
print("Todo listo")

## 3. Detectar jugadores (la parte larga: 1 a 3 horas para un partido completo)

In [ ]:
%cd /content/aro/procesador
!python pipeline.py --video "$VIDEO_PANORAMICA" --out "$CARPETA_SALIDA" --modelo yolo11s.pt --lote 16

### Control de la calibración
Las **líneas rojas** tienen que caer sobre el borde de la cancha. Si no coinciden (otra cancha u otro encuadre), escribile a Claude con esta imagen.

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(CARPETA_SALIDA, "calib_vis.jpg"), width=1100))

## 4. Seguimiento, identidad y métricas

In [ ]:
extra = f"--inicio-aprox {INICIO_APROX}" if INICIO_APROX >= 0 else ""
!python procesar.py --partido "$RESPALDO_ARO" --dets "$CARPETA_SALIDA/dets.json" --feats "$CARPETA_SALIDA/feats.json" --calib "$CARPETA_SALIDA/calib.json" --out "$CARPETA_SALIDA/fisico.json" $extra

## 5. Importar en Aro
Se descarga `fisico.json` (también queda guardado en tu carpeta de Drive). En Aro abrí el partido → **Análisis → Rendimiento físico → Importar análisis físico (.json)**. Ahí mismo podés revisar los tramos sin identificar.

In [ ]:
from google.colab import files
files.download(os.path.join(CARPETA_SALIDA, "fisico.json"))

## 6. (Opcional) Video panorámico con el nombre de cada jugador
Genera `panoramica_con_etiquetas.mp4` en tu carpeta de Drive: recuadro con número y nombre sobre cada jugador de tu equipo y minimapa de la cancha. Subilo a YouTube como **no listado** (por ejemplo "Vista completa con etiquetas").
Para el partido entero tarda bastante; con `HASTA` podés generar solo un tramo para probar (en segundos de la grabación, 0 = hasta el final).

In [ ]:
DESDE = 0 #@param {type:"number"}
HASTA = 300 #@param {type:"number"}
!python render.py --video "$VIDEO_PANORAMICA" --salida "$CARPETA_SALIDA" --desde $DESDE --hasta $HASTA --ancho 1920